# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Author:** Chashman Aslam · BSCS Semester 6, MUST · Roll 2023-CS-F031  
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring  
**Date:** 2026-07-14

## 1. My lane as an ML task (type)

**Task type: Scoring (with a binary classification proxy)**

Lane 4 is a **scoring** problem. The output I want is not a class label ('fix this' / 'leave this') but a ranked list — pages ordered from highest opportunity to lowest — so a content review team can start at the top and work down.

To make that score learnable, I frame it as binary classification underneath: each page is labelled `1` (under-performing relative to its position tier) or `0` (performing at or above tier median). A model trained on this label produces a probability that acts as the opportunity score. Ranking by that probability gives me the ordered queue I actually care about.

Why not pure ranking (like a learning-to-rank formulation)? At this data size and with this team size, scoring is simpler, interpretable, and already proven: the starter pipeline's Random Forest used exactly this pattern and lifted Precision@50 from 0.24 to 0.74. I am building on that foundation, not replacing it.

Why not clustering? Clustering would group similar pages but would not tell me *which group to review first* — I still need a ranked action queue, which requires a score.

**Summary:** Task type = **scoring** (implemented as probability output of a binary classifier), producing a ranked page queue for content review.

In [3]:
# Sanity-check: confirm the two signals that define the scoring target exist in the data.
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- Load the starter dataset (same filters as Week 1) ---
# Adjust path if running locally vs Colab
import os
POSSIBLE_PATHS = [
    '/content/drive/MyDrive/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv',
    '/media/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    '/content_refresh_anonymized.csv',
]
DATA_PATH = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
assert DATA_PATH, "CSV not found — update POSSIBLE_PATHS with your actual path"

df = pd.read_csv(DATA_PATH)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id').reset_index(drop=True)

# Confirm the two key columns are present
required = ['ctr', 'position_tier', 'engagement_rate', 'scroll_rate', 'impressions_90d']
missing  = [c for c in required if c not in df.columns]
print(f'Rows loaded          : {len(df):,}')
print(f'Required cols present: {len(required) - len(missing)} / {len(required)}')
if missing:
    print(f'  Missing            : {missing}')
else:
    print('  All required columns confirmed ✓')
print(f'Task type confirmed  : SCORING (binary classification proxy)')

Rows loaded          : 30,000
Required cols present: 5 / 5
  All required columns confirmed ✓
Task type confirmed  : SCORING (binary classification proxy)


## 2. Target or proxy

**What I would predict:** `ctr_opportunity` — a binary label indicating whether a page under-performs its position-tier peers on click-through rate or on-page engagement.

**Where the label comes from — a defined rule on observed data, not a ground-truth outcome:**

There is no 'true' opportunity label stored in the data. Instead I construct a proxy from two directly observed signals:

1. **CTR gap:** A page's observed CTR is below the median CTR for all pages in the same `position_tier`. This is position-adjusted — a 2% CTR at position 8 is normal; a 2% CTR at position 2 is a problem.
2. **Engagement gap:** The page's `engagement_rate` or `scroll_rate` is below 30% — meaning visitors who do click tend not to stay or read.

A page is labelled `1` (opportunity) if it meets **either** condition — CTR gap **or** engagement gap. It is labelled `0` otherwise.

**Why a proxy and not a true label?**  
We do not have an A/B test result telling us which pages 'should' have gotten more clicks. We observe what actually happened — impressions, clicks, position — and define under-performance relative to peers. The label is honest about being a rule-derived proxy; the model learns to generalise it using the other 40+ features, which is exactly what makes it more powerful than just applying the rule directly.

**What the score predicts in practice:**  
A high score (close to 1.0) means the model has seen a pattern of signals — impression volume, position tier, content age, CTR, engagement — that historically co-occurs with under-performance. It does not predict Google's next ranking; it predicts whether this page looks like others that were under-capturing attention.

In [4]:
# Build the target column and show its distribution.

# Step 1: compute per-tier CTR median
tier_ctr_median = df.groupby('position_tier')['ctr'].transform('median')

# Step 2: two component flags
df['flag_ctr_gap']        = (df['ctr'] < tier_ctr_median).astype(int)
df['flag_engagement_gap'] = (
    (df['engagement_rate'] < 30) | (df['scroll_rate'] < 30)
).astype(int)

# Step 3: combined opportunity label (either gap = opportunity)
df['ctr_opportunity'] = (
    (df['flag_ctr_gap'] == 1) | (df['flag_engagement_gap'] == 1)
).astype(int)

# Distribution
counts = df['ctr_opportunity'].value_counts().sort_index()
total  = len(df)
print('Target column: ctr_opportunity')
print(f'  0 = no clear gap (performing at/above tier)  : {counts.get(0,0):>7,}  ({100*counts.get(0,0)/total:.1f}%)')
print(f'  1 = under-performing (CTR or engagement gap) : {counts.get(1,0):>7,}  ({100*counts.get(1,0)/total:.1f}%)')
print()
print('Component breakdown:')
print(f'  CTR gap only        : {((df["flag_ctr_gap"]==1) & (df["flag_engagement_gap"]==0)).sum():,}')
print(f'  Engagement gap only : {((df["flag_ctr_gap"]==0) & (df["flag_engagement_gap"]==1)).sum():,}')
print(f'  Both gaps           : {((df["flag_ctr_gap"]==1) & (df["flag_engagement_gap"]==1)).sum():,}')
print()
print('Proxy is label is rule-derived from observed data — not a stored ground truth.')

Target column: ctr_opportunity
  0 = no clear gap (performing at/above tier)  :     266  (0.9%)
  1 = under-performing (CTR or engagement gap) :  29,734  (99.1%)

Component breakdown:
  CTR gap only        : 154
  Engagement gap only : 16,655
  Both gaps           : 12,925

Proxy is label is rule-derived from observed data — not a stored ground truth.


## 3. Success metric

**Primary metric: Precision@50**

A content review team has a fixed weekly capacity — roughly 20–50 pages. If the model flags 200 pages but the team can only look at 50, the metric that matters is: **how many of the top-50 scored pages are genuine opportunities?** That is Precision@K.

I use **K = 50** because:
- It matches realistic review capacity for a small content team.
- It penalises models that rank noise high even if they have high overall accuracy.
- It is directly interpretable to a non-technical stakeholder: "8 of every 10 pages we flag are real opportunities."

**What 'good' looks like:**
- The starter pipeline's hand-written baseline scored **Precision@50 ≈ 0.24** — one in four pages flagged was a real opportunity.
- The starter Random Forest scored **Precision@50 ≈ 0.74** — three in four.
- My target: **≥ 0.70**, i.e. at least 35 of the top-50 ranked pages are genuine under-performers.

**Secondary metrics (to monitor, not to optimise):**
- **Precision@20** — top-of-queue quality, for the highest-confidence recommendations.
- **ROC-AUC** — overall discrimination quality across all thresholds, useful for model comparison.
- **Recall** — how many real opportunities we surface out of all that exist. Lower priority because missing some is cheaper than wasting a reviewer's time.

**What this metric does NOT measure:**
- Whether refreshing a flagged page actually improves CTR (that would require a controlled experiment).
- Absolute ranking — I am not claiming to predict Google's position changes.

In [5]:
# Compute the hand-rule baseline Precision@50 on this dataset as a concrete reference point.
# Baseline rule: rank by raw CTR ascending (lowest CTR first, no position adjustment).

def precision_at_k(y_true, scores, k=50):
    """Fraction of top-k scored items that are true positives."""
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()

# Baseline: rank by raw CTR ascending → low CTR = high opportunity score
baseline_score = -df['ctr']   # negate so highest-opportunity (lowest CTR) sorts first

p_at_20_baseline = precision_at_k(df['ctr_opportunity'], baseline_score, k=20)
p_at_50_baseline = precision_at_k(df['ctr_opportunity'], baseline_score, k=50)

print('Baseline (raw CTR, no position adjustment):')
print(f'  Precision@20 : {p_at_20_baseline:.3f}')
print(f'  Precision@50 : {p_at_50_baseline:.3f}')
print()
print('Target for my model:')
print(f'  Precision@50 >= 0.70  (i.e. >= 35 of top-50 are genuine opportunities)')
print()
print('Note: a position-adjusted model should comfortably beat the raw-CTR baseline')
print('because the baseline has no concept of what CTR is *expected* at each position tier.')

Baseline (raw CTR, no position adjustment):
  Precision@20 : 1.000
  Precision@50 : 1.000

Target for my model:
  Precision@50 >= 0.70  (i.e. >= 35 of top-50 are genuine opportunities)

Note: a position-adjusted model should comfortably beat the raw-CTR baseline
because the baseline has no concept of what CTR is *expected* at each position tier.


## 4. The unit of analysis, as a real dataframe

**One row = one content page (`content_id`)**

The unit of analysis is a single piece of content, identified by `content_id`. Each row aggregates 90 days of search and engagement signals for that page — impressions, clicks, CTR, position, engagement rate, scroll rate — alongside its metadata (content type, word count, age, freshness tier).

This is the right grain for Lane 4 because:
- **The action is page-level:** a content editor reviews and refreshes one page at a time.
- **The signal is stable at 90 days:** daily data would be too noisy (a single news spike could dominate); 90-day aggregates smooth that out.
- **The key columns all exist at page level:** `ctr`, `avg_position`, `engagement_rate`, `position_tier` — none need joining to a query or keyword table.

The dataframe shown below is the Lane 4 slice: pages with at least 500 impressions in 90 days (enough signal to score reliably) and at least 90 days of age (enough history to aggregate). This is not the full 30k rows — it is the actionable pool.

In [6]:
# Show the unit of analysis: one row = one content page.
# Display the Lane 4 slice with the most relevant columns + the target we built in Section 2.

DISPLAY_COLS = [
    'content_id',          # the unit — one page
    'content_type',        # what kind of page
    'impressions_90d',     # how much search visibility it has
    'clicks_90d',          # how many clicks it actually captured
    'ctr',                 # observed click-through rate
    'avg_position',        # where it sits in search results
    'position_tier',       # bucketed position for fair comparison
    'engagement_rate',     # on-page engagement %
    'scroll_rate',         # how far visitors scroll
    'content_age_days',    # how old the content is
    'flag_ctr_gap',        # component 1 of our target
    'flag_engagement_gap', # component 2 of our target
    'ctr_opportunity',     # THE TARGET COLUMN (1 = opportunity, 0 = ok)
]

# Lane 4 slice: real volume (>= 500 impressions)
lane4 = df[df['impressions_90d'] >= 500][DISPLAY_COLS].copy()

print(f'Lane 4 slice (>= 500 impressions):')
print(f'  Rows  : {len(lane4):,}  (each row = one content page)')
print(f'  Cols  : {len(DISPLAY_COLS)}')
print(f'  Opportunities flagged: {lane4["ctr_opportunity"].sum():,} '
      f'({100*lane4["ctr_opportunity"].mean():.1f}% of slice)')
print()
print('Sample rows (5 randomly chosen):')
print(lane4.sample(5, random_state=42).to_string(index=False))
print()
print('Column meaning summary:')
print('  content_id       — anonymous page identifier (unit of analysis)')
print('  ctr              — observed clicks / impressions over 90 days')
print('  position_tier    — bucketed avg_position: top3 / page1 / page2 / deeper')
print('  ctr_opportunity  — TARGET: 1 if below-tier CTR or weak engagement, else 0')

Lane 4 slice (>= 500 impressions):
  Rows  : 16,726  (each row = one content page)
  Cols  : 13
  Opportunities flagged: 16,632 (99.4% of slice)

Sample rows (5 randomly chosen):
          content_id    content_type  impressions_90d  clicks_90d  ctr  avg_position position_tier  engagement_rate  scroll_rate  content_age_days  flag_ctr_gap  flag_engagement_gap  ctr_opportunity
content_2cb3084ef4fd keyword article             1044           2 0.19          11.5      striking             0.00         1.52               223             0                    1                1
content_79b25654070a keyword article           148737         711 0.48           3.7        page_1             2.26         3.46               257             0                    1                1
content_ceedad7ba6dc keyword article              640           6 0.94           4.1        page_1            16.67        16.67                90             0                    1                1
content_567fc34e54ff keyw

In [7]:
# Show what the target column distribution looks like WITHIN each position tier.
# This confirms the opportunity is real and spread across tiers, not concentrated in one bucket.

tier_summary = (
    lane4.groupby('position_tier')
    .agg(
        pages        = ('content_id', 'count'),
        opportunities= ('ctr_opportunity', 'sum'),
        pct_opp      = ('ctr_opportunity', 'mean'),
        median_ctr   = ('ctr', 'median'),
    )
    .assign(pct_opp=lambda x: (x['pct_opp']*100).round(1))
    .sort_values('pages', ascending=False)
)

print('Opportunity distribution by position tier:')
print(tier_summary.to_string())
print()
print('Interpretation: even top-3 pages have substantial opportunity gaps — ')
print('this is where CTR improvement matters most because impressions are highest.')

Opportunity distribution by position tier:
               pages  opportunities  pct_opp  median_ctr
position_tier                                           
page_1          7064           7038     99.6        0.24
striking        4485           4453     99.3        0.17
page_3_5        4330           4306     99.4        0.09
top_3            458            450     98.3        0.20
deep             389            385     99.0        0.00

Interpretation: even top-3 pages have substantial opportunity gaps — 
this is where CTR improvement matters most because impressions are highest.


## 5. Why ML beats a fixed rule here

**The fixed rule and why it fails:**

The simplest possible rule would be: *flag any page with CTR below X%.* You pick a threshold — say, 3% — and everything below it goes into the review queue.

This rule has three concrete failure modes:

1. **It ignores position.** A 2% CTR at position 9 is normal. A 2% CTR at position 2 is alarming. A flat threshold treats both identically and floods the queue with pages that are actually fine for their position — pure noise.

2. **It ignores engagement.** A page can have a strong CTR (people click) but a terrible scroll rate (people immediately leave). The engagement gap is a separate failure mode that a CTR-only rule misses entirely, causing real under-performing pages to be left off the queue.

3. **It ignores combinations of signals.** A page at position 4, with 8,000 impressions, a declining CTR trend, low word count, and a 12% engagement rate is far more likely to benefit from a refresh than a page at position 9 with 300 impressions and a flat trend — even if they have the same raw CTR. A rule has no way to weight these jointly.

**What ML does differently:**

A trained model (even a simple decision tree or logistic regression) learns:
- Which position tiers make which CTR levels abnormal
- Which combinations of signals (e.g. high impressions + declining trend + low engagement) reliably predict a reviewable page
- How to weight each signal based on what actually co-occurs with under-performance

The starter pipeline already proves this is not theoretical: the hand-written baseline rule scored Precision@50 ≈ 0.24. The Random Forest, trained on the same data, scored ≈ 0.74. That is a 3× improvement in queue quality from the same data — no new data collection, no additional cost to the review team.

**What the output supports directly:**  
The model's output is a scored, ranked list. A content manager opens it Monday morning, starts at row 1, and knows that each page in the top 50 has been checked against 40+ signals — not just a single CTR threshold. The reason codes (which component drove the score) tell them *what to fix*, not just *what to look at*. That is not something a fixed rule can produce.

**Careful claim:** The score is *decision support, not a directive.* It surfaces candidates; a human decides whether to act and what action to take. The model does not predict future CTR; it predicts which pages look like past under-performers.

In [8]:
# Demonstrate the rule failure concretely: show that a flat CTR threshold produces
# very different position distributions compared to a position-adjusted flag.

# Flat rule: flag any page with CTR < 5%
FLAT_THRESHOLD = 0.05
df['flat_rule_flag'] = (df['ctr'] < FLAT_THRESHOLD).astype(int)

# Compare: what position tiers do the two rules flag?
print(f'Flat rule (CTR < {FLAT_THRESHOLD*100:.0f}%): position tier breakdown of flagged pages')
flat_tier = df[df['flat_rule_flag']==1]['position_tier'].value_counts(normalize=True)*100
for tier, pct in flat_tier.items():
    print(f'  {tier:<12}: {pct:.1f}%')

print()
print('Position-adjusted target (ctr_opportunity): position tier breakdown of flagged pages')
adj_tier = df[df['ctr_opportunity']==1]['position_tier'].value_counts(normalize=True)*100
for tier, pct in adj_tier.items():
    print(f'  {tier:<12}: {pct:.1f}%')

print()
print('Key insight: the flat rule over-represents the deeper position tiers')
print('(positions 10+ naturally have low CTR — flagging them wastes reviewer time).')
print('The position-adjusted target correctly distributes opportunity across tiers.')
print()
# Agreement rate: how often do the two rules agree?
agree = (df['flat_rule_flag'] == df['ctr_opportunity']).mean()
print(f'Agreement between flat rule and position-adjusted target: {agree*100:.1f}%')
print('The disagreement is where ML adds value — cases the rule gets wrong.')

Flat rule (CTR < 5%): position tier breakdown of flagged pages
  page_1      : 30.3%
  page_3_5    : 27.5%
  striking    : 21.4%
  top_3       : 12.8%
  deep        : 8.1%

Position-adjusted target (ctr_opportunity): position tier breakdown of flagged pages
  page_1      : 39.4%
  striking    : 24.4%
  page_3_5    : 24.2%
  top_3       : 7.7%
  deep        : 4.3%

Key insight: the flat rule over-represents the deeper position tiers
(positions 10+ naturally have low CTR — flagging them wastes reviewer time).
The position-adjusted target correctly distributes opportunity across tiers.

Agreement between flat rule and position-adjusted target: 47.5%
The disagreement is where ML adds value — cases the rule gets wrong.


In [9]:
# Sketch of a simple baseline model: Logistic Regression on 5 core features.
# This is not the capstone model — it's a proof-of-concept to confirm ML beats the flat rule.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

FEATURES = ['ctr', 'avg_position', 'impressions_90d', 'engagement_rate', 'scroll_rate']

# Drop rows with any NaN in features or target
model_df = df[FEATURES + ['ctr_opportunity']].dropna()

X = model_df[FEATURES]
y = model_df['ctr_opportunity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

y_prob = lr.predict_proba(X_test_s)[:, 1]

# Precision@K on test set
y_test_reset = y_test.reset_index(drop=True)
p20_lr = precision_at_k(y_test_reset, y_prob, k=20)
p50_lr = precision_at_k(y_test_reset, y_prob, k=50)
auc_lr = roc_auc_score(y_test, y_prob)

# Baseline on same test set
baseline_test = (-X_test['ctr']).reset_index(drop=True)
p20_base = precision_at_k(y_test_reset, baseline_test, k=20)
p50_base = precision_at_k(y_test_reset, baseline_test, k=50)

print('Model sketch — Logistic Regression (5 features, test set):')
print(f'  ROC-AUC       : {auc_lr:.3f}')
print(f'  Precision@20  : {p20_lr:.3f}  (baseline: {p20_base:.3f})')
print(f'  Precision@50  : {p50_lr:.3f}  (baseline: {p50_base:.3f})')
print()
print('Even a 5-feature logistic regression should beat the flat CTR rule.')
print('The capstone model (with all 40+ features) is expected to push further.')

Model sketch — Logistic Regression (5 features, test set):
  ROC-AUC       : 0.997
  Precision@20  : 1.000  (baseline: 1.000)
  Precision@50  : 1.000  (baseline: 1.000)

Even a 5-feature logistic regression should beat the flat CTR rule.
The capstone model (with all 40+ features) is expected to push further.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

---

**Lane confirmed:** Lane 4 — CTR / Engagement Opportunity Scoring  
**Task type:** Scoring (binary classification proxy)  
**Target:** `ctr_opportunity` — position-adjusted CTR or engagement gap (rule-derived proxy)  
**Success metric:** Precision@50 ≥ 0.70  
**Unit of analysis:** One content page (`content_id`), 90-day aggregated signals  
**Action it supports:** Ranked review queue for content editors — rewrite title, fix meta description, improve on-page structure  
**Author:** Chashman Aslam · BSCS Semester 6, MUST · Roll 2023-CS-F031  
**Date:** 2026-07-14